# Final Project Model
Our data comes from the [DPD Incidents](https://live-durhamnc.opendata.arcgis.com/documents/7132216432df4957830593359b0c4030/about) dataset.

## Data Cleaning
The code block below cleans the data.
1. Load data from excel sheet.
2. Filter for only 2023 and 2023 incidents.
3. Remove "(blank)" entries from all columns except for weapons.
4. Drop incidents with a status of unfounded or closed.
5. Aggregate data by concatenating, choosing the first value, or taking the mean.

In [3]:
import numpy as np 
import pandas as pd
from pathlib import Path

downloads_path = Path("C:/Users/aidan/Downloads")

df_incidents = pd.read_excel(downloads_path / "dpd_incidents_(ucr_nibrs_reporting).xlsx")

In [4]:
year = df_incidents["Report Date"].str.split("/").str[-1].str.strip()
df_23_24 = df_incidents[(year == "2023") | (year == "2024")].reset_index(drop=True)

for col in df_23_24.columns:
    if col != "Weapon":
        df_23_24 = df_23_24[df_23_24[col].astype(str).str.strip() != "(blank)"].reset_index(drop=True)

df_cleaned = df_23_24[(df_23_24["Status"]!="Unfounded") & (df_23_24["Status"]!="Closed (Non-Criminal)")].reset_index(drop=True)

agg_dict = {
    "UCR Code": lambda x: ", ".join(x.astype(str)),
    "ATT/COM": lambda x: ", ".join(x.astype(str)),
    "Sequence": lambda x: ", ".join(x.astype(str)),
    "Weapon": lambda x: ", ".join(x.astype(str)),
    "Offense": lambda x: ", ".join(x.astype(str)),
    "X": "mean",
    "Y": "mean", 
    "Report Date": "first",
    "Report Time": "first",
    "Status": "first",
    "Address": "first",
    "District": "first",
    "Beat": "first",
    "Tract": "first",
    "Premise": "first"
}

df_aggregated = df_cleaned.groupby("Case Number").agg(agg_dict).reset_index(drop=True)

df_cleaned.to_parquet("./cleaned.parquet")

## Modeling
Make a SVM that predicts location (X and Y), time (month of year), and severity (UCR Code) based on other features. 

First, the unique values for each column were written to a file. This was done to identify any additional cleaning and also what columns needed preprocessing.

Prior to creating the model, the following steps were completed:
1. Extract the month from the report date.
2. One-hot encode everything else.
3. Drop case number, address, and report time. 
4. Aggregate the dataframe.

In [5]:
for col in df_cleaned.columns:
    with open("./data.txt", "a") as file:
        file.write(f"{col}: {df_cleaned[col].unique()}")

In [10]:
from sklearn.preprocessing import OneHotEncoder

df_cleaned["Report Month"] = df_cleaned["Report Date"].astype(str).str.split("/").str[0]
df_svm = df_cleaned.drop(["Address", "Report Date", "Report Time"], axis=1).reset_index(drop=True)
df_svm = df_svm.dropna().reset_index(drop=True)

to_encode = ["Status", "Sequence", "ATT/COM", "UCR Code", "Offense", "District", "Beat", "Tract", "Premise", "Weapon", "Report Month"]
ohe = OneHotEncoder(categories="auto", sparse_output=False)
oh_encoded = ohe.fit_transform(df_svm[to_encode])

columns = []
for col, cats in zip(to_encode, ohe.categories_):
    for cat in cats:
        columns.append(f"{col}_{cat}") 

df_ohe = pd.DataFrame(oh_encoded, columns=columns)
df_svm_ohe = pd.concat([df_svm, df_ohe], axis=1)
df_svm = df_svm_ohe.drop(to_encode, axis=1).reset_index(drop=True)

for col in df_svm.columns:
    df_svm[col] = df_svm[col].astype(int)

In [ ]:
agg_dict = {}
for col in columns:
    agg_dict[col] = lambda x: 1 if (x == 1).any() else 0

agg_dict["X"] = "mean"
agg_dict["Y"] = "mean"

df_svm_agg = df_svm.groupby("Case Number").agg(agg_dict).reset_index(drop=True)
df_svm_agg.to_parquet("./svm_agg.parquet")

Since we are predicting an output instead of doing a classification, we use SVR and we also use MultiOutputRegressor to simplify the process of predicting the different encoded categories.

In [ ]:
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

y_col = []
for col, cats in zip(to_encode, ohe.categories_):
    if col in ["X", "Y", "UCR Code", "Report Month"]:
        for cat in cats:
            y_col.append(f"{col}_{cat}") 

X = df_svm_agg[list(set(df_svm_agg.columns) - set(y_col))]
y = df_svm_agg[y_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

svr = SVR(epsilon=0.2)
mor = MultiOutputRegressor(svr)

mor = mor.fit(X_train, y_train)

y_pred = mor.predict(X_test)

mse_one = mean_squared_error(y_test[:,0], y_pred[:,0])
mse_two = mean_squared_error(y_test[:,1], y_pred[:,1])
print(f'MSE for first regressor: {mse_one} - second regressor: {mse_two}')
mae_one = mean_absolute_error(y_test[:,0], y_pred[:,0])
mae_two = mean_absolute_error(y_test[:,1], y_pred[:,1])
print(f'MAE for first regressor: {mae_one} - second regressor: {mae_two}')